# Purview Copilot — Silver + Gold logic (self-contained)

Transformation logic is inline (no `Common/transforms.py`). **Spark session uses `Common/bootstrap`** (same as Purview pipelines — required on Windows).

**Spark code** = `build_silver` / `build_gold_fact` below (what production uses).

**Demo on local Windows:** if the PySpark worker crashes (common in `c:\\spark-dev`), set **`EXECUTE_SPARK = False`** — notebook shows **`Notebook/demo_data/*.tsv`** expected output. Run with **`EXECUTE_SPARK = True`** on Fabric or when `spark.range(1).count()` works.

**DATA_SOURCE:** `demo` | `local_parquet` | `lakehouse`

In [ ]:
import os
import sys
from pathlib import Path

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["MEDALLION_FRESH_SPARK"] = "1"

# demo | local_parquet | lakehouse
DATA_SOURCE = "demo"

# --- local_parquet: output of NB_Purview_Copilot_BronzeToSilver (pandas CSV loader) ---
BRONZE_PARQUET_DIR = Path.home() / ".fabric" / "merge_staging" / "_break" / "purview_copilot_bronze"

# --- lakehouse tables (Okta + HC; bronze only if DATA_SOURCE == lakehouse) ---
BRONZE_PURVIEW = "Bronze.purview_copilot_usage"
SILVER_OKTA = "Silver.okta_user_for_ai"
SILVER_HC = "Silver.hc_silver_historical"

MIN_CREATION_DATE = "2026-09-01"  # prod: 2026-10-01
CREATION_MONTH = "2026-09"  # None = no extra month filter

COPILOT_ASSET_ID = "PENDING"
COPILOT_ASSET_NAME = "M365 Copilot"
COMPARE_OLD_INNER_JOIN = True  # False in demo to run faster
SHOW_ROWS = 20
DEMO_DATA_DIR = Path("Notebook/demo_data")
# demo: False = show bundled TSV only (no Spark jobs). True = run build_silver/build_gold.
EXECUTE_SPARK = False if DATA_SOURCE == "demo" else True

In [ ]:
_cwd = Path.cwd()
if not (_cwd / "Common" / "bootstrap.py").is_file():
    _repo = _cwd.parent
    if (_repo / "Common" / "bootstrap.py").is_file():
        os.chdir(_repo)
if DATA_SOURCE == "demo":
    COMPARE_OLD_INNER_JOIN = False
%run ./Common/bootstrap
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
from Common.bootstrap import _stop_active_spark, create_onelake_spark

_stop_active_spark()
spark = create_onelake_spark("Purview-Logic")
spark.conf.set("spark.sql.shuffle.partitions", "4")
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "false")
if DATA_SOURCE == "demo":
    spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "104857600")
print(f"Spark {spark.version} python={sys.executable}", flush=True)
if EXECUTE_SPARK:
    try:
        spark.range(1).count()
        print("Spark worker probe: OK", flush=True)
    except Exception as _probe_err:
        EXECUTE_SPARK = False
        print(f"Spark worker probe FAILED — forcing EXECUTE_SPARK=False ({_probe_err})", flush=True)
else:
    print("EXECUTE_SPARK=False — demo bundled output only (Spark code cells are still the spec)", flush=True)

In [ ]:
import shutil

from pyspark.sql import DataFrame, functions as F

PREVIEW_PARQUET = Path.home() / ".fabric" / "logic_preview" / "preview.parquet"

OKTA_COLS = [
    "OktaUserId", "OktaUserEmail", "OktaUserFullEmail",
    "OktaUserEmployeeNumber", "OktaUserFullName", "OktaUserRegion", "OktaUserCountry",
]


def _bronze_col(df: DataFrame, *names: str):
    lower = {c.lower(): c for c in df.columns}
    for n in names:
        if n.lower() in lower:
            return F.col(lower[n.lower()])
    raise ValueError(f"Missing column (any of): {names}")


def email_fallback(email: F.Column) -> F.Column:
    return F.when(email.rlike(r"^10[^@]*@"), F.substring(email, 3, 512))


def prompt_count_from_audit_json(messages_json: F.Column) -> F.Column:
    """Count \"isPrompt\":true in Messages JSON (string ops only — no lambda filter / broken regexp_count)."""
    blob = F.coalesce(messages_json, F.lit(""))
    sentinel = "\u0001PROMPT\u0001"
    marked = F.regexp_replace(blob, r'"isPrompt"\s*:\s*true', sentinel)
    return F.greatest(F.size(F.split(marked, sentinel)) - F.lit(1), F.lit(0)).cast("int")


def _okta_for_join(okta: DataFrame, broadcast_okta: bool) -> DataFrame:
    return F.broadcast(okta) if broadcast_okta else okta


def join_okta_inner_old(base: DataFrame, okta: DataFrame, email_col: str, *, broadcast_okta: bool = False) -> DataFrame:
    okta = _okta_for_join(okta, broadcast_okta)
    u = F.col(email_col)
    a = base.join(okta, u == okta.OktaUserEmail, "inner")
    b = base.join(okta, u == okta.OktaUserFullEmail, "inner")
    return a.unionByName(b).dropDuplicates(["purview_copilot_usage_record_id"])


def join_okta_left_with_fallback(
    base: DataFrame, okta: DataFrame, email_col: str, fallback_col: str, *, broadcast_okta: bool = False
) -> DataFrame:
    okta = _okta_for_join(okta, broadcast_okta)
    user, fb = F.col(email_col), F.col(fallback_col)

    def _coalesce_prefixed(df: DataFrame, prefixes: list[str]) -> DataFrame:
        out = df
        for name in OKTA_COLS:
            out = out.withColumn(name, F.coalesce(*[F.col(f"{p}{name}") for p in prefixes]))
        drop = [f"{p}{c}" for p in prefixes for c in OKTA_COLS]
        return out.drop(*[c for c in drop if c in out.columns])

    p1 = okta.select(*[F.col(c).alias(f"_p1_{c}") for c in OKTA_COLS])
    p2 = okta.select(*[F.col(c).alias(f"_p2_{c}") for c in OKTA_COLS])
    step = base.join(p1, user == F.col("_p1_OktaUserEmail"), "left")
    step = step.join(p2, user == F.col("_p2_OktaUserFullEmail"), "left")
    step = _coalesce_prefixed(step, ["_p1_", "_p2_"])
    f1 = okta.select(*[F.col(c).alias(f"_f1_{c}") for c in OKTA_COLS])
    f2 = okta.select(*[F.col(c).alias(f"_f2_{c}") for c in OKTA_COLS])
    step = step.join(f1, fb == F.col("_f1_OktaUserEmail"), "left")
    step = step.join(f2, fb == F.col("_f2_OktaUserFullEmail"), "left")
    for name in OKTA_COLS:
        step = step.withColumn(
            name,
            F.when(F.col("OktaUserId").isNotNull(), F.col(name)).otherwise(
                F.coalesce(F.col(f"_f1_{name}"), F.col(f"_f2_{name}"))
            ),
        )
    drop_fb = [f"_f1_{c}" for c in OKTA_COLS] + [f"_f2_{c}" for c in OKTA_COLS]
    return step.drop(*[c for c in drop_fb if c in step.columns])


def interaction_type(app_identity: F.Column, agent_name: F.Column) -> F.Column:
    return (
        F.when(app_identity.like("Copilot.Fabric.%"), F.lit("Power BI / Fabric"))
        .when(app_identity.like("Copilot.Studio.%"), F.lit("Copilot Studio Agent"))
        .when(agent_name.isin("Analyst", "Researcher"), F.lit("Specialized Agent"))
        .when(app_identity.like("MicrosoftAgent.%"), F.lit("Built-in Agent"))
        .when(app_identity.isin("Copilot.M365Copilot.WebChat", "Copilot.M365Copilot.Bizchat"), F.lit("Web Chat"))
        .when(app_identity == "Copilot.M365Copilot.Apps", F.lit("Microsoft 365 App"))
        .otherwise(F.lit("Other"))
    )


def _hc_null_columns(joined: DataFrame) -> DataFrame:
    return (
        joined.withColumn("hc_business_unit", F.lit(None).cast("string"))
        .withColumn("hc_job_title", F.lit(None).cast("string"))
        .withColumn("hc_division", F.lit(None).cast("string"))
    )


def attach_hc(joined: DataFrame, hc: DataFrame | None) -> DataFrame:
    # Demo: pass hc=None (never call rdd.isEmpty/count — crashes Windows PySpark workers)
    if hc is None:
        return _hc_null_columns(joined)
    hc_s = hc.select(
        F.col("HCKeyEmail"),
        F.col("HCBusinessUnit").alias("hc_business_unit"),
        F.col("HCJobTitle").alias("hc_job_title"),
        F.col("HCDivision").alias("hc_division"),
    )
    hc_alt = hc_s.select(
        F.col("HCKeyEmail").alias("_hc_alt_key"),
        F.col("hc_business_unit").alias("_hc_alt_bu"),
        F.col("hc_job_title").alias("_hc_alt_job"),
        F.col("hc_division").alias("_hc_alt_div"),
    )
    out = joined.join(hc_s, F.col("purview_copilot_usage_key_email") == hc_s.HCKeyEmail, "left")
    out = out.join(hc_alt, F.col("purview_copilot_usage_key_full_email") == hc_alt._hc_alt_key, "left")
    return (
        out.withColumn("hc_business_unit", F.coalesce("hc_business_unit", "_hc_alt_bu"))
        .withColumn("hc_job_title", F.coalesce("hc_job_title", "_hc_alt_job"))
        .withColumn("hc_division", F.coalesce("hc_division", "_hc_alt_div"))
        .drop("HCKeyEmail", "_hc_alt_key", "_hc_alt_bu", "_hc_alt_job", "_hc_alt_div")
    )


def build_silver(
    bronze: DataFrame,
    okta: DataFrame,
    hc: DataFrame | None,
    min_creation_date: str,
    *,
    use_old_okta_inner_join: bool = False,
    broadcast_okta: bool = False,
) -> DataFrame:
    audit = _bronze_col(bronze, "AuditData", "audit_data").cast("string")
    user_id = _bronze_col(bronze, "UserId", "user_id")
    creation_date = F.to_timestamp(F.trim(_bronze_col(bronze, "CreationDate", "creation_date"))).cast("date")
    messages_json = F.get_json_object(audit, "$.CopilotEventData.Messages")
    app_identity = F.get_json_object(audit, "$.AppIdentity")
    agent_name = F.get_json_object(audit, "$.AgentName")

    base = (
        bronze.filter(creation_date >= F.lit(min_creation_date))
        .select(
            _bronze_col(bronze, "RecordId", "record_id").alias("purview_copilot_usage_record_id"),
            creation_date.alias("purview_copilot_usage_date"),
            F.lower(F.trim(user_id)).alias("purview_copilot_usage_email"),
            app_identity.alias("purview_copilot_usage_app_identity"),
            agent_name.alias("purview_copilot_usage_agent_name"),
            messages_json.alias("_messages_json"),
        )
        .withColumn("purview_copilot_usage_prompt_count", prompt_count_from_audit_json(F.col("_messages_json")))
        .drop("_messages_json")
        .dropDuplicates(["purview_copilot_usage_record_id"])
    )
    base = base.withColumn("purview_copilot_usage_email_fallback", email_fallback(F.col("purview_copilot_usage_email")))
    base = base.withColumn(
        "purview_copilot_usage_interaction_type",
        interaction_type(F.col("purview_copilot_usage_app_identity"), F.col("purview_copilot_usage_agent_name")),
    ).withColumn(
        "purview_copilot_usage_month_no",
        (F.year("purview_copilot_usage_date") * 100 + F.month("purview_copilot_usage_date")).cast("int"),
    )
    okta_s = okta.select(
        F.col("OktaUserId"),
        F.lower(F.trim(F.col("OktaUserEmail"))).alias("OktaUserEmail"),
        F.lower(F.trim(F.col("OktaUserFullEmail"))).alias("OktaUserFullEmail"),
        F.col("OktaUserEmployeeNumber"), F.col("OktaUserFullName"),
        F.col("OktaUserRegion"), F.col("OktaUserCountry"),
    )
    if use_old_okta_inner_join:
        joined = join_okta_inner_old(base, okta_s, "purview_copilot_usage_email", broadcast_okta=broadcast_okta)
    else:
        joined = join_okta_left_with_fallback(
            base, okta_s, "purview_copilot_usage_email", "purview_copilot_usage_email_fallback", broadcast_okta=broadcast_okta
        )
        joined = joined.dropDuplicates(["purview_copilot_usage_record_id"])
    joined = joined.withColumn(
        "purview_copilot_usage_key_email",
        F.when(F.col("OktaUserEmail").isNotNull(), F.concat(F.col("purview_copilot_usage_month_no").cast("string"), F.lit("/"), F.col("OktaUserEmail"))),
    ).withColumn(
        "purview_copilot_usage_key_full_email",
        F.when(F.col("OktaUserFullEmail").isNotNull(), F.concat(F.col("purview_copilot_usage_month_no").cast("string"), F.lit("/"), F.col("OktaUserFullEmail"))),
    )
    joined = attach_hc(joined, hc)
    return joined.select(
        "purview_copilot_usage_record_id", "purview_copilot_usage_date",
        "purview_copilot_usage_email", "purview_copilot_usage_email_fallback",
        "purview_copilot_usage_prompt_count", "purview_copilot_usage_interaction_type",
        "purview_copilot_usage_month_no",
        F.col("OktaUserId").alias("okta_user_id"),
        F.col("OktaUserEmail").alias("okta_user_email"),
        F.col("OktaUserFullEmail").alias("okta_user_full_email"),
        F.col("OktaUserFullName").alias("okta_user_full_name"),
        "hc_business_unit", "hc_job_title", "hc_division",
    )


def build_gold_fact(silver: DataFrame) -> DataFrame:
    grouped = (
        silver.groupBy("purview_copilot_usage_date", "okta_user_id", "purview_copilot_usage_interaction_type")
        .agg(
            F.first("purview_copilot_usage_month_no", ignorenulls=True).cast("int").alias("Month_No"),
            F.sum("purview_copilot_usage_prompt_count").cast("int").alias("Request"),
            F.first("okta_user_email", ignorenulls=True).alias("userEmail"),
            F.first("okta_user_full_email", ignorenulls=True).alias("userFullEmail"),
            F.first("okta_user_full_name", ignorenulls=True).alias("username"),
            F.first("hc_business_unit", ignorenulls=True).alias("User_UN"),
            F.first("hc_job_title", ignorenulls=True).alias("JobRoles"),
        )
        .filter(F.col("Request") >= 1)
    )
    use_case = (
        F.when(F.col("purview_copilot_usage_interaction_type") == "Web Chat", "UC-9007-COP-001")
        .when(F.col("purview_copilot_usage_interaction_type") == "Microsoft 365 App", "UC-9007-COP-002")
        .when(F.col("purview_copilot_usage_interaction_type") == "Built-in Agent", "UC-9007-COP-003")
        .when(F.col("purview_copilot_usage_interaction_type") == "Copilot Studio Agent", "UC-9007-COP-004")
        .when(F.col("purview_copilot_usage_interaction_type") == "Specialized Agent", "UC-9007-COP-005")
        .when(F.col("purview_copilot_usage_interaction_type") == "Power BI / Fabric", "UC-9007-COP-006")
        .otherwise("UC-9007-COP-007")
    )
    return grouped.select(
        "Month_No", "Request", F.lit(COPILOT_ASSET_ID).alias("assetId"), F.lit(COPILOT_ASSET_NAME).alias("assetName"),
        F.col("okta_user_id").alias("userId"), "username", "userEmail", "userFullEmail",
        use_case.alias("useCaseCode"), F.col("purview_copilot_usage_interaction_type").alias("useCaseDisplayName"),
        F.lit(1).alias("UserWithActivity"), F.col("purview_copilot_usage_date").alias("LastModification"),
        "User_UN", "JobRoles",
    )


def _demo_fallback_preview(name: str) -> None:
    path = DEMO_DATA_DIR / name
    if path.is_file():
        print(f"Bundled demo preview ({path.name}):\n")
        print(path.read_text(encoding="utf-8"))
    else:
        print(f"No bundled preview at {path}", flush=True)


def display_df(df: DataFrame, n: int = SHOW_ROWS, *, demo_fallback: str | None = None) -> None:
    if demo_fallback and DATA_SOURCE == "demo" and not EXECUTE_SPARK:
        _demo_fallback_preview(demo_fallback)
        return
    try:
        text = df.limit(int(n))._jdf.showString(int(n), 120, False)
        if text:
            print(text)
            return
    except Exception:
        pass
    if demo_fallback and DATA_SOURCE == "demo":
        print("Live Spark preview unavailable on this machine.", flush=True)
        _demo_fallback_preview(demo_fallback)
        return
    raise RuntimeError("Spark preview failed (set EXECUTE_SPARK=False for demo bundled output)")


def distinct_count(df: DataFrame, col: str) -> int:
    return int(df.select(col).distinct().count())

In [ ]:
if DATA_SOURCE == "demo" and not EXECUTE_SPARK:
    bronze = okta = hc = None
    print("DATA_SOURCE=demo (bundled TSV — set EXECUTE_SPARK=True to run Spark)")
elif DATA_SOURCE == "demo":
    _demo = DEMO_DATA_DIR.resolve()
    bronze = spark.read.json(str(_demo / "purview_logic_bronze.json"))
    okta = spark.read.json(str(_demo / "purview_logic_okta.json"))
    hc = None
    print(f"DATA_SOURCE=demo (read.json from {_demo})")

elif DATA_SOURCE == "local_parquet":
    if not BRONZE_PARQUET_DIR.is_dir():
        raise FileNotFoundError(
            f"Bronze parquet not found: {BRONZE_PARQUET_DIR}\n"
            "Run NB_Purview_Copilot_BronzeToSilver with PURVIEW_BRONZE_SOURCE=local_csv first."
        )
    bronze = spark.read.parquet(str(BRONZE_PARQUET_DIR))
    okta = spark.table(SILVER_OKTA)
    hc = spark.table(SILVER_HC)
    print(f"DATA_SOURCE=local_parquet from {BRONZE_PARQUET_DIR}")

elif DATA_SOURCE == "lakehouse":
    bronze = spark.table(BRONZE_PURVIEW)
    okta = spark.table(SILVER_OKTA)
    hc = spark.table(SILVER_HC)
    print(f"DATA_SOURCE=lakehouse bronze={BRONZE_PURVIEW}")
else:
    raise ValueError("DATA_SOURCE must be demo | local_parquet | lakehouse")

if CREATION_MONTH and bronze is not None:
    c = F.to_timestamp(F.trim(_bronze_col(bronze, "CreationDate", "creation_date"))).cast("date")
    start, end = F.lit(f"{CREATION_MONTH}-01"), F.add_months(F.lit(f"{CREATION_MONTH}-01"), 1)
    bronze = bronze.filter((c >= start) & (c < end))

In [ ]:
print("Silver (NEW left + fallback):")
silver_new = None
if DATA_SOURCE == "demo" and not EXECUTE_SPARK:
    _demo_fallback_preview("silver_new_preview.tsv")
    print("distinct okta_user_id — OLD inner: 1 | NEW left+fallback: 2  (expected)", flush=True)
else:
    _bc_okta = DATA_SOURCE == "demo"
    silver_new = build_silver(
        bronze, okta, hc, MIN_CREATION_DATE, use_old_okta_inner_join=False, broadcast_okta=_bc_okta
    )
    display_df(
        silver_new.select(
            "purview_copilot_usage_email", "purview_copilot_usage_email_fallback",
            "okta_user_id", "okta_user_email", "purview_copilot_usage_prompt_count",
        ),
        demo_fallback="silver_new_preview.tsv",
    )
    if COMPARE_OLD_INNER_JOIN:
        try:
            silver_old = build_silver(
                bronze, okta, hc, MIN_CREATION_DATE, use_old_okta_inner_join=True, broadcast_okta=_bc_okta
            )
            print(
                "distinct okta_user_id — OLD inner:", distinct_count(silver_old, "okta_user_id"),
                "| NEW left+fallback:", distinct_count(silver_new, "okta_user_id"),
            )
        except Exception as exc:
            print(f"OLD inner compare skipped ({exc}). Demo expects OLD=1 NEW=2.", flush=True)

In [ ]:
print("Gold fact_copilot_active:")
if DATA_SOURCE == "demo" and not EXECUTE_SPARK:
    _demo_fallback_preview("gold_preview.tsv")
    print("distinct userId: 2 | distinct userEmail: 2  (expected)", flush=True)
else:
    gold = build_gold_fact(silver_new)
    display_df(
        gold.select("userId", "userEmail", "userFullEmail", "Request", "LastModification", "useCaseDisplayName"),
        demo_fallback="gold_preview.tsv",
    )
    try:
        print("distinct userId:", distinct_count(gold, "userId"), "| distinct userEmail:", distinct_count(gold, "userEmail"))
    except Exception as exc:
        print(f"Counts skipped ({exc}). Demo expects userId=2 userEmail=2.", flush=True)